# 아폴로 11호 1202 경보 시뮬레이션

코어 세트 8개, VAC 5개인 실행 관리자 모형에 레이더 간섭 부하를 걸어 1202 경보와 재시작 보호를 재현합니다. 단순화한 교육용 모형입니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/apollo_1202_executive.ipynb)

Colab에서 CPU 런타임을 선택하고 위에서부터 셀을 실행하세요. 첫 코드 셀에서 `QUICK = True`로 바꾸면 축소 조건으로 실행합니다.

In [ ]:
QUICK = False

import random
from collections import Counter
from dataclasses import dataclass


# 아래 상수는 역사 자료의 측정값이 아닌 교육용 가정값이다.
TICK_MS = 10
CORE_SETS = 8
VAC_SLOTS = 5
PERCENT = 100
SEED = 1202
RADAR_LOADS = (0, 5, 10, 13, 15, 20)
FOCUS_LOAD = 13
FULL_SECONDS = 600
QUICK_SECONDS = 240
SAMPLE_SECONDS = 10
SERVICE_PERIOD = 200
SERVICE_TICKS = 130
LANDING_TICKS = 170
REDUCED_TICKS = 110
CHECKPOINT_TICKS = 10
SWITCH_AFTER_RESTARTS = 2
MAX_ESSENTIAL_HEADROOM = 10

TICKS_PER_SECOND = 1000 // TICK_MS
SAMPLE_TICKS = SAMPLE_SECONDS * TICKS_PER_SECOND


@dataclass(frozen=True)
class Task:
    name: str
    period: int
    cost: int
    priority: int  # 작은 숫자가 먼저 실행된다.
    essential: bool = False
    needs_vac: bool = False


ATTITUDE = Task("자세 제어", 10, 1, 0, essential=True)
HIGH_TASKS = (
    ATTITUDE,
    Task("표시 갱신", 20, 1, 1),
    Task("입력 확인", 20, 1, 2),
    Task("상태 보고", 20, 1, 3),
)
SERVICE = Task("SERVICER", SERVICE_PERIOD, SERVICE_TICKS, 4,
               essential=True, needs_vac=True)


@dataclass
class Job:
    task: Task
    released: int
    remaining: int
    checkpoint: int

    @classmethod
    def start(cls, task, tick, cost):
        return cls(task, tick, cost, cost)

    def run_tick(self):
        self.remaining -= 1
        if self.remaining % CHECKPOINT_TICKS == 0:
            self.checkpoint = self.remaining


@dataclass
class Result:
    alarms: list
    restarts: int
    stop_tick: int | None
    switch_tick: int | None
    sample_peaks: list
    service_released: int
    service_completed: int
    attitude_missed: int
    attitude_max_delay: int | None
    optional_discarded: int


def simulate(seconds, radar_load, protected, service_cost=SERVICE_TICKS,
             essential_only=False, reduce_cost=False):
    """각 틱에서 주기 작업을 등록한 뒤 인터럽트나 작업 하나를 처리한다."""
    # 선점과 자원 할당은 틱 경계에서만 처리하고 재시작에는 CPU 시간을 부과하지 않는다.
    end = seconds * TICKS_PER_SECOND
    tasks = (ATTITUDE, SERVICE) if essential_only else (*HIGH_TASKS, SERVICE)
    active = []
    alarms = []
    restarts = 0
    stop_tick = None
    switch_tick = None
    sample_peaks = [0] * ((end + SAMPLE_TICKS - 1) // SAMPLE_TICKS)
    service_released = service_completed = 0
    attitude_released = attitude_on_time = 0
    attitude_max_delay = None
    optional_discarded = 0
    # 고정 시드로 시작 위상만 정한다. 같은 r은 매 100틱에 정확히 r틱을 쓴다.
    radar_phase = random.Random(SEED).randrange(PERCENT)

    for tick in range(end):
        sample = tick // SAMPLE_TICKS
        service_due = tick % SERVICE.period == 0
        service_released += service_due
        attitude_released += tick % ATTITUDE.period == 0
        if stop_tick is not None:
            # 정지한 큐는 그대로 남고, 예정된 필수 작업은 계속 센다.
            sample_peaks[sample] = max(sample_peaks[sample], len(active))
            continue

        if (reduce_cost and service_due and switch_tick is None
                and restarts >= SWITCH_AFTER_RESTARTS):
            switch_tick = tick
            service_cost = REDUCED_TICKS

        due = [Job.start(task, tick, service_cost if task is SERVICE else task.cost)
               for task in tasks if tick % task.period == 0]
        for index, job in enumerate(due):
            # 두 자원이 모두 부족하면 코어 세트 부족을 먼저 보고한다.
            if len(active) == CORE_SETS:
                alarm = 1202
            elif job.task.needs_vac and sum(j.task.needs_vac for j in active) == VAC_SLOTS:
                alarm = 1201
            else:
                active.append(job)
                continue

            alarms.append((tick, alarm))
            sample_peaks[sample] = max(sample_peaks[sample], len(active))
            if not protected:
                stop_tick = tick
                break

            restarts += 1
            # 이번 틱에 예정됐던 작업도 포함해 가장 최근 항법 작업을 고른다.
            candidates = active + due[index:]
            optional_discarded += sum(not j.task.essential for j in candidates)
            latest_service = max((j for j in candidates if j.task is SERVICE),
                                 key=lambda j: j.released, default=None)
            active = [j for j in candidates if j.task is ATTITUDE or j is latest_service]
            for resumed in active:
                resumed.remaining = resumed.checkpoint
            break

        sample_peaks[sample] = max(sample_peaks[sample], len(active))
        if stop_tick is not None:
            continue

        radar_phase += radar_load
        if radar_phase >= PERCENT:
            radar_phase -= PERCENT
            continue
        if not active:
            continue

        # 동순위 작업은 먼저 도착한 작업부터 한 틱씩 선점 가능하게 실행한다.
        running = min(active, key=lambda j: (j.task.priority, j.released))
        running.run_tick()
        if running.remaining != 0:
            continue
        active.remove(running)
        if running.task is SERVICE:
            service_completed += 1
        elif running.task is ATTITUDE:
            delay = tick + 1 - running.released
            attitude_on_time += delay <= ATTITUDE.period
            attitude_max_delay = max(attitude_max_delay or 0, delay)

    return Result(alarms, restarts, stop_tick, switch_tick, sample_peaks,
                  service_released, service_completed,
                  attitude_released - attitude_on_time, attitude_max_delay,
                  optional_discarded)


def time_text(tick):
    if tick is None:
        return "없음"
    return f"{tick / TICKS_PER_SECOND:.2f}초"


def verdict(passed):
    return "PASS" if passed else "FAIL"


def first_alarm(result):
    if not result.alarms:
        return "없음"
    tick, code = result.alarms[0]
    return f"{time_text(tick)} ({code})"

seconds = QUICK_SECONDS if QUICK else FULL_SECONDS
print(f"시드 {SEED} | 조건별 {seconds}초 | 1틱 {TICK_MS}ms")

In [ ]:
# Chart language and style
import os
import platform
import sys
from pathlib import Path
from urllib.request import urlopen

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.text import Text

CHART_LANG = os.environ.get('CHART_LANG', 'ko')
CHART_COLORS = ('#2a78d6', '#eb6834', '#1baf7a')
CHART_TEXT = '#0b0b0b'
CHART_MUTED = '#52514e'
CHART_GRID = '#e4e3df'
CHART_BG = '#fcfcfb'
CHART_LIMIT = '#e34948'

in_colab = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
chart_font = matplotlib.rcParamsDefault['font.family']
if CHART_LANG == 'ko':
    # NanumGothic is licensed under the SIL OFL 1.1.
    nanum = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    try:
        if not nanum.is_file():
            nanum = Path('fonts/NanumGothic-Regular.ttf')
            if not nanum.is_file():
                nanum.parent.mkdir(exist_ok=True)
                url = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
                with urlopen(url, timeout=60) as response:
                    nanum.write_bytes(response.read())
        font_manager.fontManager.addfont(str(nanum))
        chart_font = font_manager.FontProperties(fname=str(nanum)).get_name()
    except Exception as exc:
        CHART_LANG = 'en'
        reason = ' '.join(str(exc).splitlines()) or type(exc).__name__
        print(f'차트 언어 en, 한글 글꼴 설정 실패, {reason}')
    else:
        print(f'차트 언어 ko, 글꼴 {chart_font}')
else:
    CHART_LANG = 'en'

matplotlib.rcParams.update({
    'figure.figsize': (8, 4.5),
    'figure.dpi': 200,
    'savefig.dpi': 200,
    'figure.facecolor': CHART_BG,
    'savefig.facecolor': CHART_BG,
    'axes.facecolor': CHART_BG,
    'axes.edgecolor': CHART_MUTED,
    'axes.labelcolor': CHART_TEXT,
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
    'axes.unicode_minus': False,
    'axes.prop_cycle': matplotlib.rcsetup.cycler(color=CHART_COLORS),
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': CHART_GRID,
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'text.color': CHART_TEXT,
    'xtick.color': CHART_MUTED,
    'ytick.color': CHART_MUTED,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'legend.frameon': False,
    'font.family': chart_font,
})

chart_location = 'Colab' if in_colab else platform.system()
CPU_RUNTIME = f'{chart_location} CPU ({platform.machine()}), Python {platform.python_version()}'


def finish_chart(fig, runtime=CPU_RUNTIME, top=1):
    for text in fig.findobj(Text):
        label = text.get_text()
        if '·' in label:
            text.set_text(label.replace(' ·', ',').replace('·', ','))
    runtime = runtime.replace(' ·', ',').replace('·', ',')
    fig.tight_layout(rect=(0, 0.065, 1, top))
    fig.text(0.02, 0.025, runtime, fontsize=8, color=CHART_MUTED)
    plt.show()

## 1. 레이더 부하와 재시작 보호

레이더 인터럽트가 CPU 시간을 쓰게 하고, 보호 설정별로 경보 횟수, 항법 작업 완료율, 자세 제어 누락을 비교합니다. 1202는 코어 세트 부족, 1201은 VAC 부족을 뜻합니다.

In [ ]:
def experiment_one(seconds):
    print("\n## 실험1 레이더 부하와 재시작 보호")
    print("r% 보호 1202 1201 재시작 코어  완료% 지연ms   정지s 누락")
    results = {}
    for load in RADAR_LOADS:
        for protected in (True, False):
            result = simulate(seconds, load, protected)
            results[load, protected] = result
            counts = Counter(code for _, code in result.alarms)
            completion = PERCENT * result.service_completed / result.service_released
            stopped = (seconds - result.stop_tick / TICKS_PER_SECOND
                       if result.stop_tick is not None else 0)
            delay = (str(result.attitude_max_delay * TICK_MS)
                     if result.attitude_max_delay is not None else "-")
            protection = "있음" if protected else "없음"
            print(f"{load:2} {protection} {counts[1202]:4} {counts[1201]:4} "
                  f"{result.restarts:6} {max(result.sample_peaks):4} "
                  f"{completion:6.1f} {delay:>6} {stopped:7.2f} "
                  f"{result.attitude_missed:4}")
    print("지연은 완료된 자세 제어 작업의 등록~완료 시간이다.")
    print("정지 시간은 첫 경보로 멈춘 시점부터 종료까지다.")
    print("SERVICER 완료율은 전체 예정 횟수 대비 완료 횟수다.")

    on = results[FOCUS_LOAD, True]
    off = results[FOCUS_LOAD, False]
    safe = all(r.attitude_missed == 0 and r.stop_tick is None
               and r.restarts == len(r.alarms)
               for (_, protected), r in results.items() if protected)
    print("누락은 자세 제어 작업이 다음 주기까지 끝나지 않은 횟수다.")
    print(f"r={FOCUS_LOAD}% 재시작 시 비필수 작업 폐기: {on.optional_discarded}개")
    passed = (safe and not results[0, True].alarms and not results[0, False].alarms
              and on.optional_discarded > 0 and off.stop_tick is not None
              and off.restarts == 0 and off.attitude_missed > 0)
    print(f"판정 {verdict(passed)}")
    return passed, on, off, results

one, on, off, radar_results = experiment_one(seconds)

In [ ]:
L = {
    'ko': {
        'title': '레이더 부하별 SERVICER 완료율', 'x': '레이더 부하 (%)', 'y': '완료율 (%)',
        True: '재시작 보호 있음', False: '재시작 보호 없음', 'source': '실행 관리자 모형',
    },
    'en': {
        'title': 'SERVICER completion by radar load', 'x': 'Radar load (%)', 'y': 'Completion rate (%)',
        True: 'With restart protection', False: 'Without restart protection', 'source': 'Executive simulation',
    },
}
labels = L[CHART_LANG]
fig, ax = plt.subplots()
for protected, marker in ((True, 'o'), (False, 's')):
    cases = [radar_results[load, protected] for load in RADAR_LOADS]
    completion = [PERCENT * case.service_completed / case.service_released for case in cases]
    ax.plot(RADAR_LOADS, completion, marker=marker, label=labels[protected])
ax.set(title=labels['title'], xlabel=labels['x'], ylabel=labels['y'])
ax.set_xticks(RADAR_LOADS)
ax.set_ylim(-5, PERCENT + 5)
ax.legend(loc='lower left')
finish_chart(fig, f"{CPU_RUNTIME} · {labels['source']}")

## 2. 시간에 따른 코어 세트 점유

같은 레이더 부하에서 구간별 최대 점유를 비교하고, 첫 경보 직후 보호 설정에 따라 큐가 어떻게 달라지는지 봅니다.

In [ ]:
def experiment_two(on, off):
    print(f"\n## 실험2 r={FOCUS_LOAD}%의 코어 세트 점유")
    print(f"보호 있음 첫 경보: {first_alarm(on)}")
    print(f"보호 없음 첫 경보: {first_alarm(off)}")
    print("구간(초)   최대 점유(보호 있음/없음)")
    for index, (on_peak, off_peak) in enumerate(zip(on.sample_peaks, off.sample_peaks)):
        start = index * SAMPLE_SECONDS
        print(f"{start:3}~{start + SAMPLE_SECONDS:<3}     {on_peak:2} / {off_peak:2}")
    print("각 구간은 끝 시각을 제외한다. 정지 후 점유는 고정된다.")
    passed = False
    if on.alarms and off.alarms:
        tick, code = on.alarms[0]
        sample = tick // SAMPLE_TICKS
        passed = (code == 1202 and on.alarms[0] == off.alarms[0]
                  and on.sample_peaks[0] < CORE_SETS
                  and on.sample_peaks[sample] == CORE_SETS
                  and all(peak == CORE_SETS for peak in off.sample_peaks[sample:]))
    print(f"판정 {verdict(passed)}")
    return passed

two = experiment_two(on, off)

In [ ]:
L = {
    'ko': {
        'title': '레이더 부하 {load}%에서의 코어 세트 점유', 'x': '경과 시간 (초)', 'y': '구간별 최대 점유',
        True: '재시작 보호 있음', False: '재시작 보호 없음',
        'limit': '코어 세트 한계', 'alarm': '첫 경보 {code} · {time:.2f}초',
        'source': '{sample}초 구간으로 집계',
    },
    'en': {
        'title': 'Core set occupancy at {load}% radar load', 'x': 'Elapsed time (s)', 'y': 'Peak sets in use per interval',
        True: 'With restart protection', False: 'Without restart protection',
        'limit': 'Core set limit', 'alarm': 'First alarm {code} at {time:.2f} s',
        'source': '{sample}-second intervals',
    },
}
labels = L[CHART_LANG]
fig, ax = plt.subplots()
for protected, case in ((True, on), (False, off)):
    edges = [index * SAMPLE_SECONDS for index in range(len(case.sample_peaks))] + [seconds]
    ax.step(edges, [*case.sample_peaks, case.sample_peaks[-1]], where='post',
            label=labels[protected])
ax.axhline(CORE_SETS, color=CHART_LIMIT, linestyle='--', linewidth=1.2, label=labels['limit'])
if on.alarms:
    tick, code = on.alarms[0]
    alarm_time = tick / TICKS_PER_SECOND
    ha = 'left' if alarm_time < seconds / 2 else 'right'
    ax.annotate(labels['alarm'].format(code=code, time=alarm_time),
                (alarm_time, CORE_SETS), xytext=(0, 12), textcoords='offset points',
                ha=ha, arrowprops={'arrowstyle': '-', 'color': CHART_MUTED})
ax.set(title=labels['title'].format(load=FOCUS_LOAD), xlabel=labels['x'], ylabel=labels['y'],
       xlim=(0, seconds), ylim=(0, CORE_SETS + 2))
ax.legend(loc='lower right')
sample_note = labels['source'].format(sample=SAMPLE_SECONDS)
finish_chart(fig, f'{CPU_RUNTIME} · {sample_note}')

## 3. 착륙 직전의 긴 항법 계산

SERVICER 계산 시간을 늘립니다. 필수 작업만 있는 조건과 표시 갱신 등 나머지 작업도 있는 조건에서 어떤 자원이 먼저 부족해지는지 비교합니다.

In [ ]:
def experiment_three(seconds):
    print("\n## 실험3 착륙 직전 장시간 계산")
    essential_load = PERCENT * (LANDING_TICKS / SERVICE.period + ATTITUDE.cost / ATTITUDE.period)
    headroom = PERCENT - essential_load
    print(f"필수 작업 CPU 여유: {headroom:.1f}% (인터럽트 제외)")
    print(f"레이더 부하 반영 후 여유: {headroom - FOCUS_LOAD:.1f}%")
    essential = simulate(seconds, FOCUS_LOAD, True, LANDING_TICKS, essential_only=True)
    full = simulate(seconds, FOCUS_LOAD, True, LANDING_TICKS)
    for label, result in (("필수만", essential), ("전체", full)):
        counts = Counter(code for _, code in result.alarms)
        print(f"{label}: 1202 {counts[1202]}, 1201 {counts[1201]}, 재시작 {result.restarts}회")
        print(f"  첫 경보 {first_alarm(result)}, 자세 제어 누락 {result.attitude_missed}회")
    passed = (0 <= headroom < MAX_ESSENTIAL_HEADROOM
              and sum(code == 1201 for _, code in essential.alarms) >= 2
              and sum(code == 1202 for _, code in full.alarms) >= 2
              and all(r.attitude_missed == 0 and r.restarts == len(r.alarms)
                      for r in (essential, full)))
    print(f"판정 {verdict(passed)}")
    return passed, full

three, baseline = experiment_three(seconds)

## 4. P66을 본뜬 항법 계산량 감소

재시작이 누적되면 SERVICER 계산량을 줄입니다. 변경 시점을 기준으로 이전과 이후의 경보를 집계하고, 계산량을 유지한 3번 실험의 같은 시간 구간과 비교합니다.

In [ ]:
def experiment_four(seconds, baseline):
    print("\n## 실험4 P66을 본뜬 SERVICER 계산량 감소")
    result = simulate(seconds, FOCUS_LOAD, True, LANDING_TICKS, reduce_cost=True)
    switch = result.switch_tick
    print(f"SERVICER 계산량: {LANDING_TICKS} → {REDUCED_TICKS}틱")
    print(f"변경 시각: {time_text(switch)}")
    passed = False
    if switch is not None:
        before = [alarm for alarm in result.alarms if alarm[0] < switch]
        baseline_before = [alarm for alarm in baseline.alarms if alarm[0] < switch]
        after = len(result.alarms) - len(before)
        baseline_after = len(baseline.alarms) - len(baseline_before)
        print(f"변경 전 경보 {len(before)}회, 변경 후 경보 {after}회")
        print(f"같은 후반 구간의 미변경 경보: {baseline_after}회")
        passed = (before == baseline_before and len(before) == SWITCH_AFTER_RESTARTS
                  and after == 0 and baseline_after > 0 and result.attitude_missed == 0)
    print(f"자세 제어 누락: {result.attitude_missed}회")
    print(f"판정 {verdict(passed)}")
    return passed

four = experiment_four(seconds, baseline)

## 판정 요약

각 실험에서 설정한 모형의 판정 조건을 확인합니다.

In [ ]:
print("\n## 요약")
print("실험  판정")
results = (one, two, three, four)
for number, passed in enumerate(results, start=1):
    print(f"{number:>4}  {verdict(passed)}")
print(f"전체  {verdict(all(results))}")